# CAVE V1DD addition comparison

Compare **mean pooling, pointwise MLP, and GraphTransformer (GT)** on paired **fold 0**, with and without V1DD training cells. No confidence cutoff is applied.

| Condition | Training cells | Held-out cells |
|---|---|---|
| Without V1DD | Existing MICrONS training cohort | Same MICrONS + V1DD test cells |
| With V1DD | Same MICrONS cohort + V1DD training cells | Same MICrONS + V1DD test cells |

Report MICrONS, V1DD, and combined metrics separately. V1DD candidate labels map ITC → BipFam, DTC → MartFam, and STC-Neurogliaform- → NglFam. These are putative family assignments. The presynaptic windows use the existing soma cut, synapse matching, and 10-node cable-distance rules.

The trainer selects checkpoints using the held-out test fold; there is no separate validation set. One fold gives a paired comparison without a cross-fold uncertainty estimate.


In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "results" / "presynaptic").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from analysis.presynaptic.cave_v1dd_addition.cave_v1dd_comparison import (
    RESULT_ROOT, ARCHITECTURES, DATASETS, LABELS, TARGET_CLASSES,
    load_training_curves, load_summaries, metric_deltas, per_class_metrics,
    plot_training_curves, plot_metric_comparison, plot_per_class_recall, plot_confusions,
)

def show_fold_figure(fig, fold):
    if fig is not None:
        title = fig._suptitle.get_text() if fig._suptitle is not None else ""
        fig.suptitle(f"Fold {fold}" + (f" — {title}" if title else ""))
        display(fig)
        plt.close(fig)

RESULT_ROOT


## Cohort and preparation

Final eligible cell and window counts are available after dataset preparation. MICrONS windows are reused; V1DD windows require completed and verified embeddings. Cells with no valid unique windows are explicitly excluded.


In [ ]:
DATASET_ROOT = Path("/orcd/scratch/orcd/013/jcbliao/presynaptic_axons/cave_v1dd_addition")
metadata_path = DATASET_ROOT / "metadata.json"
if metadata_path.exists():
    display(json.loads(metadata_path.read_text()))
else:
    print("Dataset finalization is pending.")
inventory_path = DATASET_ROOT / "window_inventory.csv"
if inventory_path.exists():
    inventory = pd.read_csv(inventory_path)
    display(inventory)


## Training progress

Separate panels compare the two training conditions for each architecture. Epoch metrics cover the combined held-out set; dataset-separated metrics appear in the final results. Re-run these cells to refresh progress.


In [ ]:
curves = load_training_curves()
if curves.empty:
    print("No completed training epochs yet; training waits for inference and window preparation.")
else:
    for fold, fold_curves in curves.groupby("fold", sort=True):
        latest = (fold_curves.sort_values("epoch")
                  .groupby(["condition", "architecture", "run"], observed=True).tail(1))
        display(latest)
        show_fold_figure(plot_training_curves(fold_curves), fold)


## Final held-out metrics

Cell metrics use majority voting across each cell's windows. Window metrics measure local predictions. Counts are shown so the paired test populations can be checked. These results load directly from each completed run's JSON; the summary job does not need to finish first.


In [ ]:
summary, payloads = load_summaries()
if summary.empty:
    print("Final results appear after training finishes.")
else:
    for fold, fold_summary in summary.groupby("fold", sort=True):
        display(fold_summary.sort_values(["dataset", "architecture", "condition"]))
        show_fold_figure(plot_metric_comparison(fold_summary), fold)


## Change from adding V1DD training cells

Positive values favor training with V1DD. Deltas compare the same architecture, dataset, and fold, and require both conditions to finish. They are absolute score changes (multiply by 100 for percentage points). The loader checks equal held-out cell and window counts.


In [ ]:
deltas = metric_deltas(summary)
if deltas.empty:
    print("Paired deltas require both training conditions for at least one architecture.")
else:
    display(deltas.sort_values(["fold", "dataset", "architecture"]))


## Per-class recall

Focus on bipolar, Martinotti, and neurogliaform candidate families in each dataset. Support is the number of true cells/windows. Unsupported true classes have undefined recall and are shown as missing; the aggregate metrics above retain the trainer's class convention.


In [ ]:
class_metrics = per_class_metrics(payloads)
if class_metrics.empty:
    print("Per-class metrics appear with final results.")
else:
    targets = class_metrics[class_metrics.cell_type.isin(TARGET_CLASSES)]
    display(targets.sort_values(["fold", "dataset", "granularity", "architecture", "cell_type", "condition"]))
    for fold, fold_metrics in class_metrics.groupby("fold", sort=True):
        for dataset in DATASETS:
            show_fold_figure(plot_per_class_recall(fold_metrics, dataset), fold)
# All classes, including precision and support, remain available in class_metrics.


## Confusion matrices

For each architecture and dataset, the first row shows window predictions and the second shows cell predictions. Columns compare training without and with V1DD. Entries report row-normalized fractions followed by raw counts. Rows with no true examples are omitted; all predicted classes remain visible so predictions outside the three V1DD families are retained.


In [ ]:
if not payloads:
    print("Confusion matrices appear with final results.")
else:
    for fold in sorted({key[0] for key in payloads}):
        for dataset in DATASETS:
            for architecture in ARCHITECTURES:
                fig = plot_confusions(payloads, fold, architecture, dataset)
                if fig is not None:
                    display(fig)
                    plt.close(fig)
